# 02. Prefix Sum + Hashing

Prefix sum alone answers range queries. Combined with a **hash map** it answers "**how many / how long subarrays** have property X" in O(n), even with negative numbers.

**Learning order:** CORE IDEA → SUBARRAY SUM = K → LONGEST SUBARRAY SUM = K → DIVISIBLE BY K → CONTIGUOUS ARRAY → INTERVIEW THINKING

## 1. Core idea

Subarray `i..j` has sum `k` when `prefix[j+1] - prefix[i] = k`. Rewrite:

`prefix[i] = prefix[j+1] - k`

So while scanning, at each position ask: "**have I seen the prefix `current - k` before?**" A hash map remembers earlier prefixes. Each earlier match is one valid subarray ending here.

Real life: "on which days did my balance rise by exactly ₹500 compared to an earlier day?" Look up (today's balance − 500) in the list of past balances.

## 2. Subarray Sum Equals K (count)

Map: `prefix value → how many times seen`. Start with `{0: 1}` because an empty prefix (before index 0) has sum 0, and it lets subarrays that start at index 0 be counted.

**Order matters:** look up `prefix - k` **first**, then store the current prefix. Otherwise `k = 0` would count the empty subarray.

In [1]:
from collections import defaultdict

def subarray_sum_equals_k(nums, k):
    seen = defaultdict(int)
    seen[0] = 1
    prefix = 0
    count = 0
    for x in nums:
        prefix += x
        count += seen[prefix - k]      # earlier prefixes that make a valid subarray
        seen[prefix] += 1
    return count

print(subarray_sum_equals_k([1, 2, 3, 3], 6))    # 2  ([1,2,3] and [3,3])
print(subarray_sum_equals_k([1, -1, 0], 0))      # 3

2
3


## 3. Longest Subarray with Sum K

Now we need the **length**, so store the **first index** where each prefix appeared (the earliest start gives the longest subarray). Do not overwrite an existing entry.

In [2]:
def longest_subarray_sum_k(nums, k):
    first = {0: -1}                 # prefix 0 "seen" before index 0
    prefix = 0
    best = 0
    for i, x in enumerate(nums):
        prefix += x
        if prefix - k in first:
            best = max(best, i - first[prefix - k])
        if prefix not in first:     # keep the earliest index
            first[prefix] = i
    return best

print(longest_subarray_sum_k([1, -1, 5, -2, 3], 3))   # 4
print(longest_subarray_sum_k([-2, -1, 2, 1], 1))      # 2

4
2


## 4. Subarrays Divisible by K

`sum(i..j)` divisible by `k` ⇔ `prefix[j+1] % k == prefix[i] % k`. So count pairs of equal **remainders**.

Python's `%` already returns a non-negative result for positive `k`, so negative numbers are fine (in C++/Java you must fix it).

In [3]:
def subarrays_div_by_k(nums, k):
    seen = defaultdict(int)
    seen[0] = 1
    prefix = 0
    count = 0
    for x in nums:
        prefix = (prefix + x) % k
        count += seen[prefix]
        seen[prefix] += 1
    return count

print(subarrays_div_by_k([4, 5, 0, -2, -3, 1], 5))   # 7

7


## 5. Contiguous Array (equal 0s and 1s)

Trick: treat every `0` as `-1`. Then "equal 0s and 1s" means **sum = 0**. That is the longest-subarray-with-sum-0 problem. Same code as section 3 with `k = 0`.

In [4]:
def find_max_length(nums):
    first = {0: -1}
    total = 0
    best = 0
    for i, x in enumerate(nums):
        total += 1 if x == 1 else -1
        if total in first:
            best = max(best, i - first[total])
        else:
            first[total] = i
    return best

print(find_max_length([0, 1]))            # 2
print(find_max_length([0, 1, 0]))         # 2
print(find_max_length([0, 0, 1, 0, 0, 0, 1, 1]))   # 6

2
2
6


## 6. Interview thinking

**Signals:** "number of subarrays with sum k", "longest subarray with ...", "divisible by k", "equal number of X and Y" (convert to ±1).

**Say:** "A subarray sum is a difference of two prefix sums. I scan once, keep the prefixes seen so far in a hash map, and for each position look up the prefix that would complete a valid subarray. O(n) time, O(n) space."

**Common mistakes:** forgetting `{0: 1}` / `{0: -1}`, storing before looking up, overwriting the first index in *longest* problems, using sliding window when negatives exist.

## Quick Revision

- Subarray sum = `prefix[j+1] - prefix[i]`. Look up `prefix - k`.
- Count → map of **counts** with `{0: 1}`. Longest → map of **first index** with `{0: -1}`.
- Lookup first, store after.
- Divisible by k → equal remainders.
- Equal 0s and 1s → map 0 to −1, sum 0.